# logit-thermo A2-11：印の大きさの確かめと、T_m ÷ T_0 のそろい方（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 15〜30 分。
- ゲート：`docs/A2_11_GATE.md`。A2-8 の読み直しで写しの印を 3・4・6 トークンに変えて n と T_m を出し直し（C）、A2-2・A2-5・A2-6 の 25 本で T_m⁺ ÷ T_0 を出して（D）判定し、`A2_11/a2_11_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：結晶の始まりの探索と A2-8 のコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_11_probe_univ.py`（`dc19161630b0e12d…`）は埋め込み（データを開く前に固定）。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo A2-11 印の大きさの確かめと T_m ÷ T_0 のそろい方（CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-11 印の大きさの確かめと T_m ÷ T_0 のそろい方（CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-11 probe and universality', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import math
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRCON': '0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9', 'SRC8': '06a93c4eec9f3e770ca9e2f3f791a94b37153c908becd0f83d825acfbd11bacd'})
SRCON, SRC8 = _s['SRCON'], _s['SRC8']
SRC11 = '"""A2-11（ゲート docs/A2_11_GATE.md）：(C) 協同性の単位 n は写しの印の大きさのせいか、(D) T_m ÷ T_0 はモデルを越えてそろうか。CPU。\n生成も読み直しもしない。scripts/a2_explore_onset.py（copy_marks）と scripts/a2_8_vh.py（phases・doc_counts・fit_vh・A8_TU・A8_SOURCES・\nA8_MODELS・A8_T0）を実行したのと同じ名前空間で実行する。データを開く前に固定する。\n\n(C) 材料：A2-8 の読み直し（`A2_8/{name}.npz`。トークンと U ＝ −log p₁ と HT）。写しの印を「g トークンの組がそれより前（続きの中）に出ていたら 1」\n    として g ＝ 3・4・6 で作り直し、A2-8 と同じ決め方（中心の窓 32、入口 0.5・出口 0.25、凍る回・溶ける回がそれぞれ 10 回以上の温度）で\n    T_m・ファントホッフの ΔH・熱量計の ΔH・n を出す。g ＝ 4 は A2-8 と同じ数になる（確かめる）。\n(D) 材料：A2-2・A2-5・A2-6 の生成（`A2_2/{name}.npz` など。温度 0.6〜1.5 の 15 個 × サンプル 2 × 記事 40 × 512 歩）、探索 8.12 と同じ 25 本。\n    写しの印 g ＝ 4 で T_m を出す。温度は 0.6 からしかないので、これを T_m⁺（0.6 以上の温度だけで出した T_m）と呼ぶ。\n    A2-8 の 3 モデルでは、0.6 以上だけにすると T_m は 0.04〜0.05 高く出た（0.567 → 0.608 など）。T_0 は各実験の結果（固定の数）。\n"""\nimport math\nimport numpy as np\n\nA11_G = [3, 4, 6]\nA11_RUNS = {\n    \'A2_2\': [\'pythia-1b\', \'qwen2.5-0.5b\', \'qwen2.5-0.5b-inst\', \'qwen2.5-1.5b\', \'olmo2-1b\', \'smollm2-1.7b\', \'smollm2-1.7b-inst\'],\n    \'A2_5\': [\'pythia-70m\', \'pythia-160m\', \'pythia-410m\', \'pythia-1b\', \'pythia-1.4b\', \'pythia-2.8b\',\n             \'pythia-1b@step8000\', \'pythia-1b@step24000\', \'pythia-1b@step64000\'],\n    \'A2_6\': [\'qwen2.5-0.5b\', \'qwen2.5-1.5b\', \'qwen2.5-3b\', \'smollm2-135m\', \'smollm2-360m\', \'smollm2-1.7b\',\n             \'qwen3-0.6b\', \'qwen3-1.7b\', \'qwen3-4b\'],\n}\nA11_T22 = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.4, 1.5]\nA11_T0 = {\'A2_2/pythia-1b\': (0.852045, 0.810311, 0.869373), \'A2_2/qwen2.5-0.5b\': (0.857873, 0.827184, 0.887921),\n          \'A2_2/qwen2.5-0.5b-inst\': (0.880881, 0.829638, 0.922522), \'A2_2/qwen2.5-1.5b\': (0.910086, 0.861524, 0.956799),\n          \'A2_2/olmo2-1b\': (0.800822, 0.775764, 0.856064), \'A2_2/smollm2-1.7b\': (0.800452, 0.756107, 0.81886),\n          \'A2_2/smollm2-1.7b-inst\': (0.814278, 0.788919, 0.857462), \'A2_5/pythia-70m\': (0.847192, 0.830408, 0.880878),\n          \'A2_5/pythia-160m\': (0.829054, 0.803702, 0.839713), \'A2_5/pythia-410m\': (0.849135, 0.81199, 0.863422),\n          \'A2_5/pythia-1b\': (0.841417, 0.81333, 0.870235), \'A2_5/pythia-1.4b\': (0.845578, 0.803406, 0.8815),\n          \'A2_5/pythia-2.8b\': (0.818831, 0.79674, 0.852551), \'A2_5/pythia-1b@step8000\': (0.848006, 0.825036, 0.874433),\n          \'A2_5/pythia-1b@step24000\': (0.82072, 0.819876, 0.8601), \'A2_5/pythia-1b@step64000\': (0.858904, 0.825706, 0.875041),\n          \'A2_6/qwen2.5-0.5b\': (0.847806, 0.832104, 0.879202), \'A2_6/qwen2.5-1.5b\': (0.925402, 0.86499, 0.972219),\n          \'A2_6/qwen2.5-3b\': (0.913934, 0.869474, 0.984563), \'A2_6/smollm2-135m\': (0.809256, 0.76964, 0.831557),\n          \'A2_6/smollm2-360m\': (0.794828, 0.75636, 0.827773), \'A2_6/smollm2-1.7b\': (0.798879, 0.759564, 0.838186),\n          \'A2_6/qwen3-0.6b\': (0.895509, 0.85676, 0.943189), \'A2_6/qwen3-1.7b\': (0.925114, 0.88927, 1.01637),\n          \'A2_6/qwen3-4b\': (0.928481, 0.901447, 1.01439)}\nA11_PYTHIA_SIZE = [(\'A2_5/pythia-70m\', 70e6), (\'A2_5/pythia-160m\', 160e6), (\'A2_5/pythia-410m\', 410e6), (\'A2_5/pythia-1b\', 1.0e9),\n                   (\'A2_5/pythia-1.4b\', 1.4e9), (\'A2_5/pythia-2.8b\', 2.8e9)]\nA11_RHO_CRIT = 0.886                  # 6 点のスピアマンの両側 5% の境（|ρ| ≥ 0.886）\nA11_CQ1_BINS = [(0.8, \'印と逆に小さくなる（0.8 倍未満）\'), (1.25, \'ほぼ動かない（0.8〜1.25 倍）\'), (1.5, \'少し動く（1.25〜1.5 倍）\'),\n                (float(\'inf\'), \'印と一緒に大きくなる（1.5 倍以上）\')]\nA11_DQ1_BINS = [(0.05, \'そろう（ばらつき 5% 未満）\'), (0.10, \'ゆるくそろう（5〜10%）\'), (float(\'inf\'), \'そろわない（10% 以上）\')]\n\n\ndef copy_marks_g(seq, g):\n    """続きのトークン列 → c。t 歩目で終わる g トークンの組が、それより前（続きの中）に出ていたら 1。g ＝ 4 は copy_marks と同じ。"""\n    s = np.asarray(seq, dtype=np.int64)\n    L = len(s)\n    c = np.zeros(L, np.int8)\n    if L < g:\n        return c\n    gr = np.stack([s[i:L - g + 1 + i] for i in range(g)], axis=1)\n    _, first, inv = np.unique(gr, axis=0, return_index=True, return_inverse=True)\n    inv = np.asarray(inv).reshape(-1)\n    c[g - 1:] = (first[inv] < np.arange(len(gr))).astype(np.int8)\n    return c\n\n\ndef build_counts_g(Z, g):\n    """A2-8 の build_counts と同じ。写しの印だけ g トークンの組にする。"""\n    nD = len(Z[\'use\'])\n    out = np.zeros((len(A8_TU), nD, 10))\n    for key, _, temps in A8_SOURCES:\n        toks, U, HT = Z[f\'{key}_toks\'], Z[f\'{key}_U\'], Z[f\'{key}_HT\']\n        for k, T in enumerate(temps):\n            ku = A8_TU.index(T)\n            for j in range(nD):\n                C = np.stack([copy_marks_g(toks[k, s, j], g) for s in range(toks.shape[1])])\n                out[ku, j] += doc_counts(C, U[k, :, j].astype(np.float64), HT[k, :, j].astype(np.float64))\n    return out\n\n\ndef counts_tokens(toks, temps, g=4):\n    """toks：(nT, S, nD, L) → (17, nD, 10)（U・HT は 0）。"""\n    nT, S, nD, L = toks.shape\n    out = np.zeros((len(A8_TU), nD, 10))\n    z = np.zeros((S, L))\n    for k, T in enumerate(temps):\n        ku = A8_TU.index(T)\n        for j in range(nD):\n            C = np.stack([copy_marks_g(toks[k, s, j], g) for s in range(S)])\n            out[ku, j] += doc_counts(C, z, z)\n    return out\n\n\ndef spearman(x, y):\n    rx, ry = np.argsort(np.argsort(x)).astype(float), np.argsort(np.argsort(y)).astype(float)\n    rx -= rx.mean(); ry -= ry.mean()\n    return float((rx @ ry) / math.sqrt((rx @ rx) * (ry @ ry)))\n\n\ndef bin_word(x, bins):\n    if not np.isfinite(x):\n        return \'はっきりしない\'\n    for e, w in bins:\n        if x < e:\n            return w\n\n\ndef analyze11(base):\n    res = {\'C\': {}, \'D\': {}}\n    # (C)\n    for name, _ in A8_MODELS:\n        Z = dict(np.load(f\'{base}/A2_8/{name}.npz\'))\n        row = {}\n        for g in A11_G:\n            p = fit_vh(build_counts_g(Z, g).sum(1))\n            row[str(g)] = {k: p.get(k) for k in (\'ok\', \'Tm\', \'dH_vH\', \'dH_cal\', \'n\', \'use_idx\')}\n        res[\'C\'][name] = row\n    ratios = [res[\'C\'][m][\'6\'][\'n\'] / res[\'C\'][m][\'3\'][\'n\'] for m, _ in A8_MODELS\n              if res[\'C\'][m][\'6\'].get(\'n\') and res[\'C\'][m][\'3\'].get(\'n\') and np.isfinite(res[\'C\'][m][\'6\'][\'n\']) and np.isfinite(res[\'C\'][m][\'3\'][\'n\'])\n              and res[\'C\'][m][\'3\'][\'n\'] > 0]\n    res[\'C_ratio_n6_n3\'] = ratios\n    res[\'CQ1\'] = bin_word(float(np.median(ratios)), A11_CQ1_BINS) if len(ratios) >= 2 else \'はっきりしない\'\n    stable = 0\n    for m, _ in A8_MODELS:\n        tms = [res[\'C\'][m][str(g)][\'Tm\'] for g in A11_G]\n        if all(t is not None and np.isfinite(t) for t in tms) and max(tms) / min(tms) <= 1.05:\n            stable += 1\n    res[\'CQ2\'] = stable\n    # (D)\n    for folder, names in A11_RUNS.items():\n        for n in names:\n            key = f\'{folder}/{n}\'\n            Z = np.load(f\'{base}/{folder}/{n}.npz\')\n            assert [float(t) for t in Z[\'temps\']] == A11_T22, key\n            p = fit_vh(counts_tokens(Z[\'toks\'], A11_T22).sum(1))\n            tm = p.get(\'Tm\') if p[\'ok\'] else float(\'nan\')\n            t0 = A11_T0[key][0]\n            res[\'D\'][key] = dict(Tm_plus=tm, dH_vH=p.get(\'dH_vH\'), use_temps=[A8_TU[i] for i in p[\'use_idx\']], T0=t0,\n                                 ratio=(tm / t0 if np.isfinite(tm) else float(\'nan\')))\n    rs = [v[\'ratio\'] for v in res[\'D\'].values() if np.isfinite(v[\'ratio\'])]\n    res[\'D_n_runs\'] = len(rs)\n    res[\'D_ratio_mean\'] = float(np.mean(rs)) if rs else float(\'nan\')\n    res[\'D_ratio_cv\'] = float(np.std(rs) / np.mean(rs)) if len(rs) >= 5 else float(\'nan\')\n    res[\'DQ1\'] = bin_word(res[\'D_ratio_cv\'], A11_DQ1_BINS) if len(rs) >= 5 else \'はっきりしない\'\n    sz = [(s, res[\'D\'][k][\'Tm_plus\']) for k, s in A11_PYTHIA_SIZE if np.isfinite(res[\'D\'][k][\'Tm_plus\'])]\n    if len(sz) == 6:\n        rho = spearman(np.log([s for s, _ in sz]), [t for _, t in sz])\n        res[\'D_pythia_rho\'] = rho\n        res[\'DQ2\'] = \'大きいほど T_m が下がる\' if rho <= -A11_RHO_CRIT else \'大きいほど T_m が上がる\' if rho >= A11_RHO_CRIT else \'はっきりしない\'\n    else:\n        res[\'D_pythia_rho\'] = float(\'nan\')\n        res[\'DQ2\'] = \'はっきりしない\'\n    return res\n\n\ndef self_test_a11():\n    rng = np.random.default_rng(0)\n    for _ in range(20):\n        sq = rng.integers(0, 6, 80)\n        assert (copy_marks_g(sq, 4) == copy_marks(sq)).all()\n    assert copy_marks_g([1, 2, 3, 1, 2, 3], 3).tolist() == [0, 0, 0, 0, 0, 1]\n    assert copy_marks_g([1, 2, 3, 1, 2, 3], 6).tolist() == [0] * 6\n    assert spearman([1, 2, 3, 4, 5, 6], [6, 5, 4, 3, 2, 1]) == -1.0\n    assert bin_word(1.0, A11_CQ1_BINS) == \'ほぼ動かない（0.8〜1.25 倍）\' and bin_word(2.0, A11_CQ1_BINS) == \'印と一緒に大きくなる（1.5 倍以上）\'\n    assert bin_word(0.03, A11_DQ1_BINS) == \'そろう（ばらつき 5% 未満）\' and bin_word(0.2, A11_DQ1_BINS) == \'そろわない（10% 以上）\'\n    assert set(A11_T0) == {f\'{f}/{n}\' for f, ns in A11_RUNS.items() for n in ns} and len(A11_T0) == 25\n    assert all(T in A8_TU for T in A11_T22)\n    return \'OK\'\n'
assert hashlib.sha256(SRC11.encode()).hexdigest() == 'dc19161630b0e12db758ad9154a3877ed1f90d4d45c5bf971bf78fe6afdfa4d0'
exec(SRCON)
exec(SRC8)
exec(SRC11)
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): o = o.item()
    if isinstance(o, float) and not math.isfinite(o): return None
    return o
print('自己テスト：', self_test_onset(), self_test_a8(), self_test_a11())


In [ ]:
# セル3：数えて判定して保存（結果は表示しない）
try:
    r = analyze11('/content/drive/MyDrive/logit-thermo')
    r.update(gate='docs/A2_11_GATE.md', onset_sha256='0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9', a2_8_sha256='06a93c4eec9f3e770ca9e2f3f791a94b37153c908becd0f83d825acfbd11bacd', a2_11_sha256='dc19161630b0e12db758ad9154a3877ed1f90d4d45c5bf971bf78fe6afdfa4d0')
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_11', exist_ok=True)
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_11' + '/a2_11_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-11で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：A2-11 の判定を A2_11/a2_11_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
